In [1]:
# Reference Links: https://aswnss.hashnode.dev/sentiment-analysis-with-naive-bayes-classifier-using-nltk-and-scikit-learn

# load the dataset
# Importing libraries
import nltk
import numpy as np
import pandas as pd
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

import pprint, time

#download the treebank corpus from nltk
nltk.download('treebank')

#download the universal tagset from nltk
nltk.download('universal_tagset')

# reading the Treebank tagged sentences
nltk_data = list(nltk.corpus.treebank.tagged_sents(tagset='universal'))

#print the first two sentences along with tags
print(nltk_data[:2])

[nltk_data] Downloading package treebank to /home/tamim/nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package universal_tagset to
[nltk_data]     /home/tamim/nltk_data...
[nltk_data]   Package universal_tagset is already up-to-date!


[[('Pierre', 'NOUN'), ('Vinken', 'NOUN'), (',', '.'), ('61', 'NUM'), ('years', 'NOUN'), ('old', 'ADJ'), (',', '.'), ('will', 'VERB'), ('join', 'VERB'), ('the', 'DET'), ('board', 'NOUN'), ('as', 'ADP'), ('a', 'DET'), ('nonexecutive', 'ADJ'), ('director', 'NOUN'), ('Nov.', 'NOUN'), ('29', 'NUM'), ('.', '.')], [('Mr.', 'NOUN'), ('Vinken', 'NOUN'), ('is', 'VERB'), ('chairman', 'NOUN'), ('of', 'ADP'), ('Elsevier', 'NOUN'), ('N.V.', 'NOUN'), (',', '.'), ('the', 'DET'), ('Dutch', 'NOUN'), ('publishing', 'VERB'), ('group', 'NOUN'), ('.', '.')]]


In [2]:
# print each word with its respective tag for first two sentences
for sent in nltk_data[:2]:
    for tuple in sent:
        print(tuple)

('Pierre', 'NOUN')
('Vinken', 'NOUN')
(',', '.')
('61', 'NUM')
('years', 'NOUN')
('old', 'ADJ')
(',', '.')
('will', 'VERB')
('join', 'VERB')
('the', 'DET')
('board', 'NOUN')
('as', 'ADP')
('a', 'DET')
('nonexecutive', 'ADJ')
('director', 'NOUN')
('Nov.', 'NOUN')
('29', 'NUM')
('.', '.')
('Mr.', 'NOUN')
('Vinken', 'NOUN')
('is', 'VERB')
('chairman', 'NOUN')
('of', 'ADP')
('Elsevier', 'NOUN')
('N.V.', 'NOUN')
(',', '.')
('the', 'DET')
('Dutch', 'NOUN')
('publishing', 'VERB')
('group', 'NOUN')
('.', '.')


In [3]:
# split data into training and validation set in the ratio 80:20
train_set,test_set =train_test_split(nltk_data,train_size=0.80,test_size=0.20,random_state = 101)

# create list of train and test tagged words
train_tagged_words = [ tup for sent in train_set for tup in sent ]
test_tagged_words = [ tup for sent in test_set for tup in sent ]
print(len(train_tagged_words))
print(len(test_tagged_words))

80310
20366


In [4]:
# use set datatype to check how many unique tags are present in training data
tags = {tag for word,tag in train_tagged_words}
print(len(tags))
print(tags)

# check total words in vocabulary
vocab = {word for word,tag in train_tagged_words}

12
{'PRT', 'PRON', 'ADV', 'NOUN', 'ADP', 'X', 'VERB', 'ADJ', '.', 'NUM', 'CONJ', 'DET'}


Compute Emission probability required for HMM


In [5]:
# compute Emission Probability
def word_given_tag(word, tag, train_bag = train_tagged_words):
    tag_list = [pair for pair in train_bag if pair[1]==tag]
    count_tag = len(tag_list)#total number of times the passed tag occurred in train_bag
    w_given_tag_list = [pair[0] for pair in tag_list if pair[0]==word]
    #now calculate the total number of times the passed word occurred as the passed tag.
    count_w_given_tag = len(w_given_tag_list)

    return (count_w_given_tag, count_tag)

Compute Transition Probability

In [6]:
# compute Transition Probability
def t2_given_t1(t2, t1, train_bag = train_tagged_words):
    tags = [pair[1] for pair in train_bag]
    count_t1 = len([t for t in tags if t==t1])
    count_t2_t1 = 0
    for index in range(len(tags)-1):
        if tags[index]==t1 and tags[index+1] == t2:
            count_t2_t1 += 1
    return (count_t2_t1, count_t1)

# creating t x t transition matrix of tags, t= no of tags
# Matrix(i, j) represents P(jth tag after the ith tag)

tags_matrix = np.zeros((len(tags), len(tags)), dtype='float32')
for i, t1 in enumerate(list(tags)):
    for j, t2 in enumerate(list(tags)):
        tags_matrix[i, j] = t2_given_t1(t2, t1)[0]/t2_given_t1(t2, t1)[1]

print(tags_matrix)

[[1.17416831e-03 1.76125243e-02 9.39334650e-03 2.50489235e-01
  1.95694715e-02 1.21330721e-02 4.01174158e-01 8.29745606e-02
  4.50097844e-02 5.67514673e-02 2.34833662e-03 1.01369865e-01]
 [1.41230067e-02 6.83371304e-03 3.69020514e-02 2.12756261e-01
  2.23234631e-02 8.83826911e-02 4.84738052e-01 7.06150308e-02
  4.19134386e-02 6.83371304e-03 5.01138950e-03 9.56719834e-03]
 [1.47401085e-02 1.20248254e-02 8.14584941e-02 3.21955010e-02
  1.19472459e-01 2.28859577e-02 3.39022487e-01 1.30721495e-01
  1.39255241e-01 2.98681147e-02 6.98215654e-03 7.13731572e-02]
 [4.39345129e-02 4.65906132e-03 1.68945398e-02 2.62344331e-01
  1.76826611e-01 2.88252197e-02 1.49133503e-01 1.25838192e-02
  2.40094051e-01 9.14395228e-03 4.24540639e-02 1.31063312e-02]
 [1.26550242e-03 6.96026310e-02 1.45532778e-02 3.23588967e-01
  1.69577319e-02 3.45482156e-02 8.47886596e-03 1.07061505e-01
  3.87243740e-02 6.32751212e-02 1.01240189e-03 3.20931405e-01]
 [1.85085520e-01 5.41995019e-02 2.57543717e-02 6.16951771e-02
  1

In [7]:
# convert the matrix to a df for better readability
#the table is same as the transition table shown in section 3 of article
tags_df = pd.DataFrame(tags_matrix, columns = list(tags), index=list(tags))
display(tags_df)

,PRT,PRON,ADV,NOUN,ADP,X,VERB,ADJ,.,NUM,CONJ,DET
PRT,0.001174,0.017613,0.009393,0.250489,0.019569,0.012133,0.401174,0.082975,0.045010,0.056751,0.002348,0.101370
PRON,0.014123,0.006834,0.036902,0.212756,0.022323,0.088383,0.484738,0.070615,0.041913,0.006834,0.005011,0.009567
ADV,0.014740,0.012025,0.081458,0.032196,0.119472,0.022886,0.339022,0.130721,0.139255,0.029868,0.006982,0.071373
NOUN,0.043935,0.004659,0.016895,0.262344,0.176827,0.028825,0.149134,0.012584,0.240094,0.009144,0.042454,0.013106
ADP,0.001266,0.069603,0.014553,0.323589,0.016958,0.034548,0.008479,0.107062,0.038724,0.063275,0.001012,0.320931
X,0.185086,0.054200,0.025754,0.061695,0.142226,0.075726,0.206419,0.017682,0.160869,0.003075,0.010379,0.056890
VERB,0.030663,0.035543,0.083886,0.110589,0.092357,0.215930,0.167956,0.066390,0.034807,0.022836,0.005433,0.133610
ADJ,0.011456,0.000194,0.005243,0.696893,0.080583,0.020971,0.011456,0.063301,0.066019,0.021748,0.016893,0.005243
.,0.002789,0.068769,0.052569,0.218539,0.092908,0.025641,0.089690,0.046132,0.092372,0.078210,0.060079,0.172192
NUM,0.026062,0.001428,0.003570,0.351660,0.037487,0.202428,0.020707,0.035345,0.119243,0.184220,0.014281,0.003570


Define Viterbi Algorithm

In [8]:
def Viterbi(words, train_bag = train_tagged_words):
    state = []
    T = list(set([pair[1] for pair in train_bag]))

    for key, word in enumerate(words):
        #initialise list of probability column for a given observation
        p = []
        for tag in T:
            if key == 0:
                transition_p = tags_df.loc['.', tag]
            else:
                transition_p = tags_df.loc[state[-1], tag]

            # compute emission and state probabilities
            emission_p = word_given_tag(words[key], tag)[0]/word_given_tag(words[key], tag)[1]
            state_probability = emission_p * transition_p
            p.append(state_probability)

        pmax = max(p)
        # getting state for which probability is maximum
        state_max = T[p.index(pmax)]
        state.append(state_max)
    return list(zip(words, state))

Testing: Randomly selecting 10 sentences for testing

In [9]:
# Let's test our Viterbi algorithm on a few sample sentences of test dataset
random.seed(1234)        #define a random seed to get same sentences when run multiple times to pick som

# choose random 10 numbers
rndom = [random.randint(1,len(test_set)) for x in range(10)]

# list of 10 sents on which we test the model
test_run = [test_set[i] for i in rndom]

# list of tagged words
test_run_base = [tup for sent in test_run for tup in sent]

# list of untagged words
test_tagged_words = [tup[0] for sent in test_run for tup in sent]


In [10]:
#Here We will only test 10 sentences to check the accuracy
#as testing the whole training set takes huge amount of time
start = time.time()
tagged_seq = Viterbi(test_tagged_words)
end = time.time()
difference = end-start

print("Time taken in seconds: ", difference)

# accuracy
check = [i for i, j in zip(tagged_seq, test_run_base) if i == j]

accuracy = len(check)/len(tagged_seq)
print('Viterbi Algorithm Accuracy: ',accuracy*100)

Time taken in seconds:  27.23047637939453
Viterbi Algorithm Accuracy:  93.77990430622009


Testing with a sample sentence

In [11]:
#Check how a sentence is tagged by Viterbi HMM POS tagger
test_sent="Will can see Marry"
pred_tags= Viterbi(test_sent.split())
print(pred_tags)
#Will and Marry are tagged as NUM as they are unknown words for Viterbi Algorithm

[('Will', 'PRT'), ('can', 'VERB'), ('see', 'VERB'), ('Marry', 'PRT')]


Category-Wise Precision, Recall and F1-score

In [12]:
# True labels (POS tags) from the test set
y_true =  [tup[1] for sent in test_run for tup in sent]
#print(y_true)

#predicted labels from test set
y_pred = [tup[1] for tup in tagged_seq]
#print(y_pred)
# Generate the classification report
print(classification_report(y_true, y_pred))

              precision    recall  f1-score   support

           .       1.00      1.00      1.00        29
         ADJ       0.78      0.88      0.82         8
         ADP       1.00      1.00      1.00        16
         ADV       0.91      0.91      0.91        11
        CONJ       1.00      1.00      1.00         4
         DET       1.00      1.00      1.00        18
        NOUN       1.00      0.89      0.94        62
         NUM       1.00      1.00      1.00         2
        PRON       1.00      1.00      1.00         5
         PRT       0.31      1.00      0.47         4
        VERB       0.97      0.91      0.94        35
           X       1.00      0.93      0.97        15

    accuracy                           0.94       209
   macro avg       0.91      0.96      0.92       209
weighted avg       0.97      0.94      0.95       209

